# Project 1: Perceptron from Scratch

**Goal:** Implement the perceptron algorithm in pure NumPy, visualize the decision boundary updating in real time, and build intuition for linear classifiers.

**What you'll build:**
1. A `Perceptron` class with `predict` and `train` methods
2. Real-time visualization of the decision boundary
3. Experiments: What happens when data isn't linearly separable?

**Rules:** No sklearn. NumPy + matplotlib only. You write the math.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import clear_output
import time

np.random.seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 6)
plt.rcParams['font.size'] = 12

## 1. Generate Data

We create a simple 2D dataset with two linearly separable classes.
This is done for you — your job starts at Section 2.

In [ ]:
def make_linearly_separable(n=100, margin=1.0, seed=42):
    """Generate 2D linearly separable data with labels +1 / -1."""
    rng = np.random.default_rng(seed)
    
    # Class +1: cluster centered at (2, 2)
    X_pos = rng.normal(loc=[2, 2], scale=0.8, size=(n // 2, 2))
    # Class -1: cluster centered at (-2, -2)
    X_neg = rng.normal(loc=[-2, -2], scale=0.8, size=(n // 2, 2))
    
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n // 2) + [-1] * (n // 2))
    
    # Shuffle
    idx = rng.permutation(n)
    return X[idx], y[idx]


def make_not_separable(n=100, seed=42):
    """Generate 2D data that is NOT linearly separable (overlapping clusters)."""
    rng = np.random.default_rng(seed)
    X_pos = rng.normal(loc=[1, 1], scale=1.5, size=(n // 2, 2))
    X_neg = rng.normal(loc=[-1, -1], scale=1.5, size=(n // 2, 2))
    X = np.vstack([X_pos, X_neg])
    y = np.array([1] * (n // 2) + [-1] * (n // 2))
    idx = rng.permutation(n)
    return X[idx], y[idx]


# Generate our training data
X_train, y_train = make_linearly_separable(n=100)
print(f"X shape: {X_train.shape}  |  y shape: {y_train.shape}")
print(f"Labels: {np.unique(y_train)}")
print(f"Class balance: +1={np.sum(y_train == 1)}, -1={np.sum(y_train == -1)}")

In [ ]:
def plot_data(X, y, title="Training Data", ax=None):
    """Scatter plot of 2D data, colored by label."""
    if ax is None:
        fig, ax = plt.subplots()
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='steelblue', marker='o', 
               label='+1', edgecolors='k', linewidth=0.5, s=50)
    ax.scatter(X[y == -1, 0], X[y == -1, 1], c='salmon', marker='s', 
               label='-1', edgecolors='k', linewidth=0.5, s=50)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(title)
    ax.legend()
    ax.set_aspect('equal')
    return ax


plot_data(X_train, y_train)
plt.tight_layout()
plt.show()

## 2. The Perceptron — Your Turn

### Quick recap

The perceptron classifies a point $\mathbf{x}$ using:

$$\hat{y} = \text{sign}(\mathbf{w} \cdot \mathbf{x} + b)$$

where $\mathbf{w}$ is the weight vector and $b$ is the bias.

**Update rule** — when a point $(\mathbf{x}_i, y_i)$ is misclassified:

$$\mathbf{w} \leftarrow \mathbf{w} + y_i \, \mathbf{x}_i$$
$$b \leftarrow b + y_i$$

That's it. No learning rate. No loss function. Just fix mistakes.

### Your task

Fill in the three methods marked `### YOUR CODE ###` below.

In [ ]:
class Perceptron:
    def __init__(self, n_features):
        """Initialize weights to zeros."""
        self.w = np.zeros(n_features)  # weight vector
        self.b = 0.0                    # bias scalar
        self.history = []               # for tracking (don't modify)
    
    def predict_single(self, x):
        """Predict the label (+1 or -1) for a single data point x.
        
        Compute: sign(w · x + b)
        Convention: sign(0) = 1
        
        Args:
            x: numpy array of shape (n_features,)
        Returns:
            int: +1 or -1
        """
        ### YOUR CODE — replace the line below ###
        raise NotImplementedError("Compute w·x + b, then return its sign")
    
    def predict(self, X):
        """Predict labels for a batch of points.
        
        Args:
            X: numpy array of shape (n_samples, n_features)
        Returns:
            numpy array of shape (n_samples,) with +1 or -1
        """
        ### YOUR CODE — replace the line below ###
        raise NotImplementedError("Apply predict_single to each row of X")
    
    def train_step(self, x, y):
        """Process one training example. Update weights if misclassified.
        
        Args:
            x: numpy array of shape (n_features,)
            y: true label, +1 or -1
        Returns:
            bool: True if this point was misclassified (and weights were updated)
        """
        ### YOUR CODE — replace the lines below ###
        # 1. Predict the label for x
        # 2. If wrong: update w and b using the perceptron update rule
        # 3. Return True if updated, False if already correct
        raise NotImplementedError("Predict, check, update if wrong")
    
    def train_epoch(self, X, y):
        """One pass through the entire dataset. (Done for you.)
        
        Returns:
            int: number of misclassifications in this epoch
        """
        mistakes = 0
        for i in range(len(y)):
            if self.train_step(X[i], y[i]):
                mistakes += 1
        # Snapshot for visualization
        self.history.append((self.w.copy(), self.b, mistakes))
        return mistakes
    
    def accuracy(self, X, y):
        """Fraction of correctly classified points. (Done for you.)"""
        preds = self.predict(X)
        return np.mean(preds == y)

### Sanity check

Run this cell to test your implementation. All 3 tests should pass.

In [ ]:
# --- Sanity checks (don't modify) ---
p_test = Perceptron(n_features=2)

# Test 1: initial weights are zero → predict_single should return +1 (sign(0) = 1)
assert p_test.predict_single(np.array([1.0, 1.0])) == 1, "sign(0) should be +1"
print("✓ Test 1 passed: predict_single handles zero weights")

# Test 2: set weights manually and check prediction
p_test.w = np.array([1.0, -1.0])
p_test.b = 0.0
assert p_test.predict_single(np.array([2.0, 1.0])) == 1   # 2 - 1 = 1 > 0
assert p_test.predict_single(np.array([1.0, 3.0])) == -1   # 1 - 3 = -2 < 0
print("✓ Test 2 passed: predict_single correct with known weights")

# Test 3: train_step should update on misclassification
p_test.w = np.array([0.0, 0.0])
p_test.b = 0.0
updated = p_test.train_step(np.array([1.0, 1.0]), -1)  # predicts +1, true is -1
assert updated == True, "Should report a mistake"
assert np.allclose(p_test.w, [-1.0, -1.0]), f"w should be [-1,-1], got {p_test.w}"
assert p_test.b == -1.0, f"b should be -1, got {p_test.b}"
print("✓ Test 3 passed: train_step updates correctly")

print("\n🎉 All sanity checks passed! Move on to training.")

## 3. Train & Visualize

Now train your perceptron and watch the decision boundary evolve.

In [ ]:
# Train the perceptron
model = Perceptron(n_features=2)

max_epochs = 50
for epoch in range(1, max_epochs + 1):
    mistakes = model.train_epoch(X_train, y_train)
    if epoch <= 5 or epoch % 10 == 0 or mistakes == 0:
        print(f"Epoch {epoch:3d}  |  mistakes: {mistakes:3d}  |  accuracy: {model.accuracy(X_train, y_train):.1%}")
    if mistakes == 0:
        print(f"\n✓ Converged in {epoch} epochs!")
        break
else:
    print(f"\n✗ Did not converge in {max_epochs} epochs.")

print(f"\nFinal weights: w = {model.w}, b = {model.b:.4f}")

In [ ]:
def plot_decision_boundary(X, y, w, b, title="Decision Boundary", ax=None):
    """Plot data points and the linear decision boundary w·x + b = 0."""
    if ax is None:
        fig, ax = plt.subplots()
    
    plot_data(X, y, title=title, ax=ax)
    
    # Decision boundary: w[0]*x1 + w[1]*x2 + b = 0  →  x2 = -(w[0]*x1 + b) / w[1]
    xlim = ax.get_xlim()
    if abs(w[1]) > 1e-10:
        x1_range = np.linspace(xlim[0] - 1, xlim[1] + 1, 200)
        x2_boundary = -(w[0] * x1_range + b) / w[1]
        ax.plot(x1_range, x2_boundary, 'k-', linewidth=2, label='boundary')
        ax.set_xlim(xlim)
        ax.set_ylim(ax.get_ylim())
    elif abs(w[0]) > 1e-10:
        x1_val = -b / w[0]
        ax.axvline(x=x1_val, color='k', linewidth=2, label='boundary')
    
    ax.legend()
    return ax


plot_decision_boundary(X_train, y_train, model.w, model.b, 
                       title=f"Final Boundary (converged)")
plt.tight_layout()
plt.show()

### Watch it learn

This replays the boundary at each epoch so you can see the perceptron correcting itself.

In [ ]:
for i, (w_snap, b_snap, mistakes) in enumerate(model.history):
    clear_output(wait=True)
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_decision_boundary(X_train, y_train, w_snap, b_snap,
                           title=f"Epoch {i+1}  |  mistakes: {mistakes}", ax=ax)
    plt.tight_layout()
    plt.show()
    time.sleep(0.4)

print("Done! The boundary stabilizes once there are 0 mistakes.")

## 4. Experiments — Explore & Break Things

These are open-ended. Try them, observe, and write your answers in the markdown cells.

### Experiment A: What if data is NOT linearly separable?

Train on the overlapping dataset. What happens?

In [ ]:
X_overlap, y_overlap = make_not_separable(n=100)
plot_data(X_overlap, y_overlap, title="Overlapping Classes")
plt.show()

# Train on this data — what happens?
model_overlap = Perceptron(n_features=2)
for epoch in range(1, 101):
    mistakes = model_overlap.train_epoch(X_overlap, y_overlap)
    if epoch <= 3 or epoch % 20 == 0:
        print(f"Epoch {epoch:3d}  |  mistakes: {mistakes}  |  acc: {model_overlap.accuracy(X_overlap, y_overlap):.1%}")
    if mistakes == 0:
        print(f"Converged in {epoch} epochs")
        break

plot_decision_boundary(X_overlap, y_overlap, model_overlap.w, model_overlap.b,
                       title="Non-separable Data — Final Boundary")
plt.show()

**Your observation:** *(Write what you see. Does it converge? Is the boundary stable?)*

...

### Experiment B: Does training order matter?

Shuffle the data differently and retrain. Do you get the same boundary?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for trial, ax in enumerate(axes):
    idx = np.random.permutation(len(y_train))
    X_shuf, y_shuf = X_train[idx], y_train[idx]
    
    m = Perceptron(n_features=2)
    for ep in range(50):
        if m.train_epoch(X_shuf, y_shuf) == 0:
            break
    
    plot_decision_boundary(X_shuf, y_shuf, m.w, m.b,
                           title=f"Shuffle {trial+1}: w={m.w.round(2)}", ax=ax)

plt.tight_layout()
plt.show()

**Your observation:** *(Are the boundaries the same? Why or why not?)*

...

### Experiment C: The margin question

Look at your final boundaries from Experiment B. Are any of them the "best" boundary? 

Think about which points are closest to the line. This is the **margin** — and the perceptron doesn't maximize it. That's the motivation for SVMs (Chapter 3).

In [ ]:
# Compute the margin: minimum distance from any point to the boundary
def compute_margin(X, y, w, b):
    """Minimum distance from any correctly-classified point to the decision boundary."""
    norm_w = np.linalg.norm(w)
    if norm_w < 1e-10:
        return 0.0
    distances = np.abs(X @ w + b) / norm_w
    return np.min(distances)

print(f"Margin of your trained model: {compute_margin(X_train, y_train, model.w, model.b):.4f}")
print("\n(In Chapter 3, you'll build an SVM that maximizes this margin.)")

## 5. Stretch Goals (Optional)

If you want to go deeper before moving to Chapter 3:

1. **Averaged perceptron** — instead of returning the final `w`, return the average of all `w` values across training. Does it generalize better?

2. **Convergence bound** — the perceptron convergence theorem says it converges in at most $(R / \gamma)^2$ updates, where $R$ = max distance of any point from origin, $\gamma$ = margin of the best separator. Compute both and check.

3. **3D data** — generate 3D clusters and train a perceptron. You can't plot the boundary as a line anymore — it's a plane. Try `plot_surface` in matplotlib.

In [ ]:
# Your stretch work here


## 6. What You Learned

Before moving on, write down (in your own words):

1. What does the perceptron **guarantee**, and what does it **not**?
2. Why is the perceptron's boundary not necessarily the "best" one?
3. What would you need to find the best boundary? *(Hint: Chapter 3)*

---

**Next project:** Add hinge loss + gradient descent to turn this into an SVM (Chapter 3).